# 🚗 YOLO License Plate Detector — Training Notebook

This notebook trains a **YOLOv11s** model to detect license plates using a Roboflow dataset.

**Key features:**
- All training outputs (checkpoints, logs, plots) are saved to **Google Drive** so nothing is lost if Colab disconnects.
- A dedicated **Resume Training** cell lets you pick up exactly where you left off from `last.pt`.
- External augmentations are assumed — all Ultralytics built-in augmentations are explicitly disabled.

---

## 0 · Helper Utilities

Small helper functions used throughout the notebook for checking paths, printing file sizes, and listing directory contents.

In [ ]:
import os
import shutil


def check_path(path: str, label: str = "") -> bool:
    """Print whether a path exists and return the result."""
    exists = os.path.exists(path)
    tag = f"[{label}] " if label else ""
    status = "✅ EXISTS" if exists else "❌ NOT FOUND"
    print(f"{tag}{status}  →  {path}")
    return exists


def print_file_size(path: str) -> None:
    """Print the size of a file in MB."""
    if os.path.isfile(path):
        size_mb = os.path.getsize(path) / (1024 * 1024)
        print(f"  📦 {os.path.basename(path):20s}  {size_mb:>8.2f} MB")
    else:
        print(f"  ⚠️  File not found: {path}")


def list_run_dir(directory: str, indent: int = 0) -> None:
    """Recursively list the contents of a directory."""
    if not os.path.isdir(directory):
        print(f"  ⚠️  Directory not found: {directory}")
        return
    prefix = "  " * indent
    for entry in sorted(os.listdir(directory)):
        full = os.path.join(directory, entry)
        if os.path.isdir(full):
            print(f"{prefix}📁 {entry}/")
            list_run_dir(full, indent + 1)
        else:
            size_mb = os.path.getsize(full) / (1024 * 1024)
            print(f"{prefix}📄 {entry}  ({size_mb:.2f} MB)")


print("✅ Helper utilities loaded.")

---
## 1 · Install Dependencies

Install the **Ultralytics** YOLO library and the **Roboflow** SDK.  
This cell is safe to re-run — pip will skip packages that are already installed.

In [1]:
!pip install -q ultralytics roboflow

import ultralytics
print(f"\n✅ Ultralytics version: {ultralytics.__version__}")

^C



[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip



✅ Ultralytics version: 8.4.60


---
## 2 · Mount Google Drive

Google Drive is the **only** persistent storage in Colab. We mount it and create a project folder where all training outputs will be saved.  
If Colab disconnects and you reconnect, just re-run this cell to remount Drive.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

# --- Persistent paths on Google Drive ---
DRIVE_PROJECT  = "/content/drive/MyDrive/YOLO_result"
RUN_NAME       = "license_plate"
DRIVE_RUN_DIR  = os.path.join(DRIVE_PROJECT, RUN_NAME)
WEIGHTS_DIR    = os.path.join(DRIVE_RUN_DIR, "weights")

os.makedirs(DRIVE_RUN_DIR, exist_ok=True)
os.makedirs(WEIGHTS_DIR, exist_ok=True)

print(f"✅ Drive mounted.")
print(f"   Project folder : {DRIVE_PROJECT}")
print(f"   Run folder     : {DRIVE_RUN_DIR}")
print(f"   Weights folder : {WEIGHTS_DIR}")

# Sanity check — make sure Drive is actually accessible
if not os.path.isdir("/content/drive/MyDrive"):
    raise RuntimeError(
        "❌ Google Drive does not appear to be mounted. "
        "Click the folder icon in the Colab sidebar and mount Drive manually, "
        "then re-run this cell."
    )

---
## 3 · Load Roboflow Dataset

**Paste your Roboflow download code in the cell below.**  
After running it you should have a `dataset` variable whose `.location` attribute points to the downloaded folder containing `data.yaml`.

In [ ]:
# ┌──────────────────────────────────────────────────────────┐
# │  PASTE YOUR ROBOFLOW DOWNLOAD CODE BELOW THIS LINE      │
# │                                                          │
# │  Example:                                                │
# │    from roboflow import Roboflow                         │
# │    rf = Roboflow(api_key="YOUR_API_KEY")                 │
# │    project = rf.workspace("...").project("...")          │
# │    version = project.version(1)                          │
# │    dataset = version.download("yolov11")                 │
# └──────────────────────────────────────────────────────────┘

# --- YOUR CODE HERE ---



# --- END YOUR CODE ---

In [ ]:
# Verify the dataset downloaded correctly
dataset_path = dataset.location
data_yaml    = os.path.join(dataset_path, "data.yaml")

print(f"Dataset path : {dataset_path}")

if not check_path(dataset_path, "Dataset folder"):
    raise FileNotFoundError(
        "❌ Dataset folder not found. "
        "Make sure your Roboflow download code ran successfully in the cell above."
    )

if not check_path(data_yaml, "data.yaml"):
    raise FileNotFoundError(
        f"❌ data.yaml not found inside {dataset_path}. "
        "The dataset may have downloaded in an unexpected format."
    )

print("\n✅ Dataset is ready for training.")

---
## 4 · Runtime & Path Checks

Verify GPU availability, disk space, and all critical paths before we start training.

In [ ]:
import torch

print("═" * 55)
print("  RUNTIME INFORMATION")
print("═" * 55)

# GPU
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem  = torch.cuda.get_device_properties(0).total_mem / (1024 ** 3)
    print(f"  🖥️  GPU          : {gpu_name}")
    print(f"  🧠 GPU Memory   : {gpu_mem:.1f} GB")
    print(f"  🔥 CUDA         : Available (v{torch.version.cuda})")
else:
    print("  ⚠️  CUDA is NOT available — training will be very slow on CPU!")
    print("       Go to Runtime → Change runtime type → select GPU.")

# Disk space
total, used, free = shutil.disk_usage("/")
print(f"  💾 Disk (free)   : {free / (1024 ** 3):.1f} GB")

# Paths
print()
print("  KEY PATHS")
print("  " + "-" * 50)
print(f"  Drive output     : {DRIVE_PROJECT}")
print(f"  Run directory    : {DRIVE_RUN_DIR}")
print(f"  Weights folder   : {WEIGHTS_DIR}")
print(f"  Dataset          : {dataset_path}")
print()
check_path(WEIGHTS_DIR, "Weights dir")
check_path(data_yaml,   "data.yaml")
print("═" * 55)

---
## 5 · Fresh Training

Train **YOLOv11s** from the official pretrained weights (`yolo11s.pt`).  
All Ultralytics built-in augmentations are **explicitly disabled** because the dataset is already augmented externally.

Outputs (checkpoints, plots, logs) are written directly to Google Drive so they survive Colab disconnects.

> ⚠️ **Do NOT run this cell if you are resuming an interrupted run.**  
> Jump to **Section 7 — Resume Training** instead.

In [ ]:
from ultralytics import YOLO

model = YOLO("yolo11s.pt")

results = model.train(
    # --- Basic Config ---
    data        = f"{dataset.location}/data.yaml",
    imgsz       = 640,
    epochs      = 200,        # ✅ 200 for 6.5K dataset (enough weight updates)
    patience    = 0,          # ✅ No early stopping
    batch       = 32,
    optimizer   = "AdamW",
    lr0         = 0.001,      # ✅ Default — well-tuned for AdamW fine-tuning
                              # (lrf, wd, freeze → all left at defaults)

    # --- Runtime Augmentation ---
    mosaic      = 1.0,
    mixup       = 0.1,
    copy_paste  = 0.1,
    close_mosaic = 10,
    fliplr      = 0.5,
    scale       = 0.5,
    translate   = 0.1,

    # --- OFF for License Plates ---
    flipud      = 0.0,
    erasing     = 0.0,
    degrees     = 0.0,
    perspective = 0.0,
    shear       = 0.0,

    # --- Project Config ---
    project     = "/content/drive/MyDrive/YOLO_runs",
    name        = "license_plate_augmented",
    exist_ok    = True,
    save        = True,
    verbose     = True,
    plots       = True,
)


# --- Print output paths ---
print("\n" + "═" * 55)
print("  TRAINING COMPLETE — OUTPUT PATHS")
print("═" * 55)

best_pt    = os.path.join(WEIGHTS_DIR, "best.pt")
last_pt    = os.path.join(WEIGHTS_DIR, "last.pt")
results_csv = os.path.join(DRIVE_RUN_DIR, "results.csv")

check_path(best_pt,     "best.pt")
check_path(last_pt,     "last.pt")
check_path(results_csv, "results.csv")
print(f"\n  📊 Plots folder  : {DRIVE_RUN_DIR}")
print("═" * 55)

KeyboardInterrupt: 

---
## 6 · Checkpoint Verification

Quick sanity check that the key checkpoint files were saved to Drive.

In [ ]:
best_pt = os.path.join(WEIGHTS_DIR, "best.pt")
last_pt = os.path.join(WEIGHTS_DIR, "last.pt")

print("═" * 55)
print("  CHECKPOINT VERIFICATION")
print("═" * 55)

all_ok = True

for label, path in [("best.pt", best_pt), ("last.pt", last_pt)]:
    if check_path(path, label):
        print_file_size(path)
    else:
        all_ok = False

print()
if all_ok:
    print("✅ All checkpoints present on Google Drive.")
else:
    print(
        "⚠️  One or more checkpoints are missing!\n"
        "    This can happen if training was interrupted before the first save.\n"
        "    Try running the Fresh Training cell (Section 5) again, or check\n"
        "    the run directory for partial outputs:"
    )
    print(f"    {DRIVE_RUN_DIR}")

print()
print("  📂 Full run directory contents:")
list_run_dir(DRIVE_RUN_DIR)
print("═" * 55)

---
## 7 · Resume Training from Checkpoint

**Use this cell after a Colab disconnect** to continue training from where it left off.

How it works:
1. We load the model directly from `last.pt` on Google Drive.
2. We call `model.train(resume=True)` — Ultralytics automatically restores the optimizer state, learning-rate scheduler, and epoch counter from the checkpoint.
3. Training continues to the original target epoch count; you do **not** need to re-specify hyperparameters.

> 🔑 If Colab disconnected, make sure you have re-run **Section 0** (helpers), **Section 1** (installs), and **Section 2** (Drive mount) before running this cell.

In [ ]:
from ultralytics import YOLO

ckpt_path = "/content/drive/MyDrive/YOLO_runs/license_plate_augmented/weights/last.pt"

if check_path(ckpt_path, "Checkpoint"):
    print_file_size(ckpt_path)
    print("\n🔄 Resuming training from last.pt ...\n")

    model   = YOLO(ckpt_path)
    results = model.train(resume=True)

    print("\n✅ Resumed training complete.")
else:
    print(
        "\n❌ No checkpoint found at the expected path.\n"
        "   This means training has not been started yet, or the run directory\n"
        "   was moved/deleted.\n\n"
        "   👉 Run the Fresh Training cell (Section 5) first to create an\n"
        "      initial checkpoint, then come back here if you need to resume."
    )

---
## 8 · Post-Training Evaluation

Run validation on the best checkpoint (`best.pt`) and print the main detection metrics.

In [ ]:
from ultralytics import YOLO

best_pt = "/content/drive/MyDrive/YOLO_runs/license_plate_augmented/weights/best.pt"

if not check_path(best_pt, "best.pt"):
    raise FileNotFoundError(
        "❌ best.pt not found. Run training first (Section 5 or 7)."
    )

model   = YOLO(best_pt)
metrics = model.val()

print("\n" + "═" * 55)
print("  VALIDATION RESULTS  (best.pt)")
print("═" * 55)

if hasattr(metrics, "box"):
    b = metrics.box
    print(f"  mAP@50       : {b.map50:.4f}")
    print(f"  mAP@50-95    : {b.map:.4f}")
    print(f"  Precision    : {b.mp:.4f}")
    print(f"  Recall       : {b.mr:.4f}")
else:
    print("  (Metrics object structure differs — printing raw results)")
    print(metrics)

print("═" * 55)

---
## 9 · Inference Test

Run the trained model on a sample image.  
Replace `SAMPLE_IMAGE` with the path to your test image (upload it to Colab or provide a Drive path).

Inference outputs (annotated images) are saved to Google Drive.

In [ ]:
from ultralytics import YOLO

best_pt = "/content/drive/MyDrive/YOLO_runs/license_plate_augmented/weights/best.pt"

if not check_path(best_pt, "best.pt"):
    raise FileNotFoundError(
        "❌ best.pt not found. Run training first (Section 5 or 7)."
    )

# ┌────────────────────────────────────────────────────────────┐
# │  SET YOUR SAMPLE IMAGE PATH BELOW                         │
# │  e.g. "/content/drive/MyDrive/test_images/car.jpg"        │
# └────────────────────────────────────────────────────────────┘
SAMPLE_IMAGE = "/content/sample_image.jpg"  # <-- REPLACE THIS

if not check_path(SAMPLE_IMAGE, "Sample image"):
    print(
        "\n⚠️  Upload a test image to Colab or update SAMPLE_IMAGE above.\n"
        "    You can drag-and-drop a file into the Colab file browser,\n"
        "    or use a path on Google Drive."
    )
else:
    model = YOLO(best_pt)

    # Save predictions to Drive
    inference_output = "/content/drive/MyDrive/YOLO_runs/license_plate_augmented/inference"
    os.makedirs(inference_output, exist_ok=True)

    results = model.predict(
        source  = SAMPLE_IMAGE,
        save    = True,
        project = inference_output,
        name    = "predictions",
        exist_ok = True,
        conf    = 0.25,
    )

    print(f"\n✅ Predictions saved to: {inference_output}/predictions/")
    list_run_dir(os.path.join(inference_output, "predictions"))

    # Display inline
    from IPython.display import Image, display
    pred_dir = os.path.join(inference_output, "predictions")
    for f in sorted(os.listdir(pred_dir)):
        if f.lower().endswith((".jpg", ".jpeg", ".png")):
            print(f"\n📸 {f}")
            display(Image(filename=os.path.join(pred_dir, f), width=640))

---
## 📝 Recovery & Reference Notes

### Checkpoint files

| File | Purpose |
|---|---|
| `last.pt` | **Resume interrupted training.** Contains model weights, optimizer state, scheduler state, and the epoch counter. Use this with `model.train(resume=True)`. |
| `best.pt` | **Best validation checkpoint.** Use this for inference and deployment — it represents the epoch with the highest mAP. |

### If Colab disconnects mid-training

1. **Reconnect** to a new Colab runtime.
2. **Re-run these cells in order:**
   - Section 0 — Helper Utilities
   - Section 1 — Install Dependencies
   - Section 2 — Mount Google Drive
   - Section 3 — Load Roboflow Dataset  *(re-download if needed)*
3. **Skip Section 5** (Fresh Training).
4. **Run Section 7** (Resume Training) — it loads `last.pt` from Drive and picks up where training stopped.

### Important: do NOT redefine hyperparameters when resuming

When calling `model.train(resume=True)`, Ultralytics restores the full training state from the checkpoint, including:
- Optimizer weights and momentum buffers
- Learning-rate scheduler step
- Current epoch number
- Original hyperparameters

Do **not** manually pass `lr0`, `optimizer`, `epochs`, etc. in the resume call unless you intentionally want to change the experiment. Passing them may override the saved state and produce unexpected results.

### Google Drive paths at a glance

```
/content/drive/MyDrive/YOLO_runs/
└── license_plate_augmented/
    ├── weights/
    │   ├── best.pt          ← best validation checkpoint
    │   └── last.pt          ← resume checkpoint
    ├── results.csv          ← epoch-by-epoch metrics
    ├── results.png          ← training curves
    ├── confusion_matrix.png
    ├── ...                  ← other plots & logs
    └── inference/
        └── predictions/     ← inference output images
```